# 05 — Matched inference-compute benchmark

Benchmark ModernBERT and a representative fine-tuned generative LLM
on the same LitSearch queries, the same 100 Qwen3 candidates, the
same title/abstract text, and the same 300-word document truncation.
Use one idle GPU at a time. ModernBERT is measured locally. The LLM
section targets your existing local OpenAI-compatible vLLM server and
is disabled until you fill in its model name and enable it.

Report median throughput across three measured runs after one warmup.
Do not compare only parameter counts; report training time, peak VRAM,
queries/s, documents/s, and per-query latency.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import json
import os
import re
import statistics
import time
from pathlib import Path

import requests
import torch
from datasets import load_dataset
from sentence_transformers.cross_encoder import CrossEncoder

from tools.modernbert_supervised_baseline import (
    read_jsonl,
    reproducibility_manifest,
    write_json,
)

ARTIFACT_DIR = Path(os.environ.get(
    "MODERNBERT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "modernbert_supervised_baseline"),
))
MODEL_DIR = ARTIFACT_DIR / "models" / "modernbert_base_dorismae_lambda_seed3407" / "final"
INPUT_PATH = REPO_ROOT / "LitSearch.title_abstract.qwen3.jsonl"
OUTPUT_PATH = ARTIFACT_DIR / "compute" / "matched_compute_benchmark.json"

DATASET_REVISION = "9573fb2"
RERANK_DEPTH = 100
BENCHMARK_QUERIES = 20
DOCUMENT_WORD_LIMIT = 300
MODERNBERT_BATCH_SIZE = 64
WARMUP_RUNS = 1
MEASURED_RUNS = 3

RUN_LLM_BENCHMARK = False
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8000/v1")
LLM_MODEL_ID = os.environ.get("LLM_MODEL_ID", "REPLACE_WITH_LLAMA_8B_FT_MODEL_ID")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "EMPTY")
LLM_PEAK_VRAM_GIB = os.environ.get("LLM_PEAK_VRAM_GIB")
REQUEST_TIMEOUT_SECONDS = 900

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Run notebook 02 first; missing {MODEL_DIR}")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
corpus = load_dataset(
    "princeton-nlp/LitSearch",
    "corpus_clean",
    split="full",
    revision=DATASET_REVISION,
)
corpus_lookup = {
    int(item["corpusid"]): f"Title: {item['title']}\nAbstract: {item['abstract']}"
    for item in corpus
}
rows = read_jsonl(INPUT_PATH)[:BENCHMARK_QUERIES]
benchmark_cases = []
for row in rows:
    candidate_ids = [int(value) for value in row["retrieved"][:RERANK_DEPTH]]
    documents = [
        " ".join(corpus_lookup[document_id].split()[:DOCUMENT_WORD_LIMIT])
        for document_id in candidate_ids
    ]
    benchmark_cases.append({"query": row["query"], "documents": documents})
assert len(benchmark_cases) == BENCHMARK_QUERIES
assert all(len(item["documents"]) == RERANK_DEPTH for item in benchmark_cases)

In [ ]:
model = CrossEncoder(str(MODEL_DIR), max_length=1024)

def run_modernbert_once():
    pairs = [
        (case["query"], document)
        for case in benchmark_cases
        for document in case["documents"]
    ]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    scores = model.predict(
        pairs,
        batch_size=MODERNBERT_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    assert len(scores) == BENCHMARK_QUERIES * RERANK_DEPTH
    return elapsed

for _ in range(WARMUP_RUNS):
    run_modernbert_once()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
modernbert_times = [run_modernbert_once() for _ in range(MEASURED_RUNS)]
modernbert_median = statistics.median(modernbert_times)
modernbert_result = {
    "measured_seconds": modernbert_times,
    "median_seconds": modernbert_median,
    "median_query_latency_ms": 1000 * modernbert_median / BENCHMARK_QUERIES,
    "queries_per_second": BENCHMARK_QUERIES / modernbert_median,
    "documents_per_second": BENCHMARK_QUERIES * RERANK_DEPTH / modernbert_median,
    "peak_vram": (
        torch.cuda.max_memory_reserved() / (1024 ** 3)
        if torch.cuda.is_available() else 0.0
    ),
}
print(json.dumps(modernbert_result, indent=2))

In [ ]:
def make_llm_prompt(query, documents):
    passages = "\n\n---\n\n".join(
        f"[{index}] {document}"
        for index, document in enumerate(documents, start=1)
    )
    return (
        "You are an expert academic paper reranker. "
        f"Your task is to re-order the given list of passages (from [1] to [{len(documents)}]) "
        "based on their relevance to the query. Respond with only the ranking and nothing else.\n\n"
        "Example output for 8 passages:\n"
        "[2] > [5] > [4] > [8] > [6] > [1] > [3] > [7]\n\n"
        f"Query: {query}\n\nPassages:\n{passages}\n\n"
        "Your ranking (most to least relevant):"
    )

def available_model_ids():
    models_url = f"{LLM_BASE_URL.rstrip('/')}/models"
    response = requests.get(
        models_url,
        headers={"Authorization": f"Bearer {LLM_API_KEY}"},
        timeout=30,
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"LLM preflight GET {models_url} returned "
            f"HTTP {response.status_code}: {response.text[:1000]}"
        )
    payload = response.json()
    model_ids = [
        str(item["id"])
        for item in payload.get("data", [])
        if isinstance(item, dict) and item.get("id")
    ]
    if not model_ids:
        raise RuntimeError(
            f"No model IDs were returned by {models_url}: {response.text[:1000]}"
        )
    return model_ids

def call_llm(case):
    chat_url = f"{LLM_BASE_URL.rstrip('/')}/chat/completions"
    response = requests.post(
        chat_url,
        headers={"Authorization": f"Bearer {LLM_API_KEY}"},
        json={
            "model": LLM_MODEL_ID,
            "messages": [{
                "role": "user",
                "content": make_llm_prompt(case["query"], case["documents"]),
            }],
            "temperature": 0,
            "max_tokens": 800,
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"LLM POST {chat_url} returned HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )
    text = response.json()["choices"][0]["message"]["content"]
    parsed = [int(value) for value in re.findall(r"\[(\d+)\]", text)]
    valid_unique = []
    for value in parsed:
        if 1 <= value <= RERANK_DEPTH and value not in valid_unique:
            valid_unique.append(value)
    return {
        "parsed_count": len(valid_unique),
        "complete_valid_permutation": len(valid_unique) == RERANK_DEPTH,
    }

def run_llm_once():
    started = time.perf_counter()
    audits = [call_llm(case) for case in benchmark_cases]
    return time.perf_counter() - started, audits

llm_result = None
if RUN_LLM_BENCHMARK:
    detected_model_ids = available_model_ids()
    print("OpenAI-compatible endpoint:", LLM_BASE_URL)
    print("Available model IDs:", detected_model_ids)
    if LLM_MODEL_ID.startswith("REPLACE_"):
        if len(detected_model_ids) == 1:
            LLM_MODEL_ID = detected_model_ids[0]
            print("Auto-selected model ID:", LLM_MODEL_ID)
        else:
            raise ValueError(
                "Set LLM_MODEL_ID to one of the available model IDs above."
            )
    if LLM_MODEL_ID not in detected_model_ids:
        raise ValueError(
            f"LLM_MODEL_ID {LLM_MODEL_ID!r} is not served; choose one of "
            f"{detected_model_ids}."
        )
    for _ in range(WARMUP_RUNS):
        run_llm_once()
    measured = [run_llm_once() for _ in range(MEASURED_RUNS)]
    llm_times = [item[0] for item in measured]
    llm_median = statistics.median(llm_times)
    llm_audits = [audit for _, audits in measured for audit in audits]
    llm_result = {
        "model_id": LLM_MODEL_ID,
        "measured_seconds": llm_times,
        "median_seconds": llm_median,
        "median_query_latency_ms": 1000 * llm_median / BENCHMARK_QUERIES,
        "queries_per_second": BENCHMARK_QUERIES / llm_median,
        "documents_per_second": BENCHMARK_QUERIES * RERANK_DEPTH / llm_median,
        "peak_vram": (
            float(LLM_PEAK_VRAM_GIB) if LLM_PEAK_VRAM_GIB else None
        ),
        "complete_format_rate": sum(
            item["complete_valid_permutation"] for item in llm_audits
        ) / len(llm_audits),
        "mean_parsed_indices": sum(
            item["parsed_count"] for item in llm_audits
        ) / len(llm_audits),
    }
    print(json.dumps(llm_result, indent=2))
else:
    print("LLM benchmark skipped. Configure the local endpoint, then set RUN_LLM_BENCHMARK=True.")

In [ ]:
benchmark_manifest = {
    "protocol": {
        "candidate_source": str(INPUT_PATH.resolve()),
        "candidate_source_name": "Qwen3",
        "dataset_revision": DATASET_REVISION,
        "queries": BENCHMARK_QUERIES,
        "documents_per_query": RERANK_DEPTH,
        "document_word_limit": DOCUMENT_WORD_LIMIT,
        "warmup_runs": WARMUP_RUNS,
        "measured_runs": MEASURED_RUNS,
        "execution": "sequential, one model at a time on an otherwise idle GPU",
    },
    "modernbert": modernbert_result,
    "generative_llm": llm_result,
    "environment": reproducibility_manifest(),
}
write_json(OUTPUT_PATH, benchmark_manifest)
print(f"Saved compute comparison to {OUTPUT_PATH}")